In [ ]:
### PACKAGES ###

import torch
import torch.nn as nn
from torch.nn.functional import softmax
import torchmetrics as tm
import torch.optim as optim

from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor, Grayscale, RandomHorizontalFlip, RandomVerticalFlip

import deeplay as dl

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import os
import pandas as pd
import numpy as np

import time 



In [ ]:
### INPUTS ###

train_dir = ""
val_dir = ""
test_dir = ""

output_folder = ""
os.makedirs(output_folder, exist_ok=True)

In [ ]:
### ENVIRONMENT REPORT ###

# File to save the report
save = False
filename = "pytorch_environment_report.txt"
file_path = os.path.join(output_folder, filename)

print("="*40)
print("PyTorch Environment Report")
print("="*40)

print(f"PyTorch version:        {torch.__version__}")
print(f"CUDA version (PyTorch): {torch.version.cuda}")
print(f"GPU available:          {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_index = 0
    print(f"GPU name:               {torch.cuda.get_device_name(gpu_index)}")
    print(f"Compute capability:     {torch.cuda.get_device_capability(gpu_index)}")
    print(f"Current device:         {torch.cuda.current_device()}")
    print(f"Total memory (GB):      {torch.cuda.get_device_properties(gpu_index).total_memory / 1e9:.2f}")

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("="*40)
print(f"Using device:      {device}")

if save:
    with open(file_path, "w") as f:
        def print_to_file(*args, **kwargs):
            print(*args, **kwargs, file=f)

        print_to_file("="*40)
        print_to_file("PyTorch Environment Report")
        print_to_file("="*40)

        print_to_file(f"PyTorch version:        {torch.__version__}")
        print_to_file(f"CUDA version (PyTorch): {torch.version.cuda}")
        print_to_file(f"GPU available:          {torch.cuda.is_available()}")

        if torch.cuda.is_available():
            gpu_index = 0
            print_to_file(f"GPU name:               {torch.cuda.get_device_name(gpu_index)}")
            print_to_file(f"Compute capability:     {torch.cuda.get_device_capability(gpu_index)}")
            print_to_file(f"Current device:         {torch.cuda.current_device()}")
            print_to_file(f"Total memory (GB):      {torch.cuda.get_device_properties(gpu_index).total_memory / 1e9:.2f}")
        
        print_to_file("="*40)    
        print_to_file(f"Using device:      {device}")
            
    print("Environment report saved to", filename)

In [ ]:
### ZooplanktoNet architecture ###

conv_base = nn.Sequential(
                        nn.Conv2d(1, 96, kernel_size=13, padding=6),
                        nn.ReLU(inplace=True),
                        nn.MaxPool2d(kernel_size=3, stride=2),

                        nn.Conv2d(96, 256, kernel_size=7, padding=3),
                        nn.ReLU(inplace=True),
                        nn.MaxPool2d(kernel_size=3, stride=2),

                        nn.Conv2d(256, 384, kernel_size=3, padding=1),
                        nn.ReLU(inplace=True),

                        nn.Conv2d(384, 384, kernel_size=3, padding=1),
                        nn.ReLU(inplace=True),

                        nn.Conv2d(384, 384, kernel_size=3, padding=1),
                        nn.ReLU(inplace=True),
                        nn.MaxPool2d(kernel_size=3, stride=2),

                        nn.Conv2d(384, 512, kernel_size=3, padding=1),
                        nn.ReLU(inplace=True),

                        nn.Conv2d(512, 512, kernel_size=3, padding=1),
                        nn.ReLU(inplace=True),

                        nn.Conv2d(512, 512, kernel_size=3, padding=1),
                        nn.ReLU(inplace=True),
                        nn.MaxPool2d(kernel_size=3, stride=2), 
                        )

# Global pooling to convert feature maps into vector
connector = dl.Layer(nn.AdaptiveAvgPool2d, output_size=1)

# Dense top
dense_top = dl.MultiLayerPerceptron(
    in_features=512,                    # has to match the outchannel
    hidden_features=[4096, 4096],
    out_features=17,                    # number of species
    out_activation=None
)


# Full model
model = dl.Sequential(
    conv_base,
    connector,
    dense_top
)

zooplanktonet_classifier = dl.Classifier(model=model,).create()

In [ ]:
### LOAD PREVIOUSLY TRAINED MODEL ###

classifier = zooplanktonet_classifier
model_path = ""

classifier.load_state_dict(torch.load(model_path, 
                                      map_location=torch.device(device),))

print(f"Model loaded from files: {model_path}")


In [ ]:
### DATASET PREPARATION ###

augmentation = False

if augmentation:
    transform = Compose([
    Resize((256, 256)),   # Resize for uniform display
    Grayscale(num_output_channels=1),  # enforce 1 channel
    RandomHorizontalFlip(p=.5), 
    RandomVerticalFlip(p=.5),
    ToTensor()
    ])
    
else:
    transform = Compose([
    Resize((256, 256)),   # Resize for uniform display
    Grayscale(num_output_channels=1),  # enforce 1 channel
    ToTensor()
])

train = ImageFolder(train_dir, transform=transform)
train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)

val = ImageFolder(val_dir, transform=transform)
val_loader = torch.utils.data.DataLoader(val, batch_size=128, shuffle=False)

test_dataset = ImageFolder(test_dir, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

In [ ]:
### Check on data format ###
# Visualize some images with labels from the validation set

def image_examples(dataset):

    fig, axs = plt.subplots(3, 6, figsize=(16, 8))
    for ax in axs.ravel():
        # Pick random image
        idx = np.random.randint(0, len(dataset))
        image,_ = dataset[idx]

        # Convert from tensor (C,H,W) -> (H,W,C)
        image_np = image.permute(1, 2, 0).numpy()

        # Show image
        ax.imshow(image_np, cmap = 'gray')
        
    plt.tight_layout()
    plt.show()

image_examples(val_loader.dataset)

In [ ]:
### DEFINE LOSS FUNCTION AND OPTIMIZER ###

classifier = zooplanktonet_classifier 

criterion = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)
#optimizer = optim.Adam(classifier.parameters(), lr=0.001, weight_decay=1e-4)

In [ ]:
### MANUAL TRAINING LOOP ###

classifier = zooplanktonet_classifier 
classifier.to(device)

print(f"Starting training on device: {device}")

epochs = 200

# Save the loss history, loss function CrossEntropyLoss
train_loss = [] # based on training data
val_loss = [] # based on validation data

train_acc_history = []
val_acc_history = [] 

# save best model based on minimum validation loss
best_loss = float("inf")

time_per_epoch = []

for epoch in range(epochs):

    # measure time 
    start_time = time.time()

    print("\n")
    print(f"Epoch {epoch+1}/{epochs}")
    print("-" * 10)

    # TRAINING #
    # Set the model to training mode
    classifier.train()

    num_batches = len(train_loader)
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    # looping over batches
    for batch_idx, data in enumerate(train_loader, start=0):

        # get the inputs and labels for each batch
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = classifier(images)
        loss = criterion(outputs, labels) # For CrossEntropyLoss
        
        loss.backward()
        optimizer.step()


        if batch_idx % 10 == 0:
            print(
                f"Batch {batch_idx}/{num_batches} loss: {loss.item():.4f}"
            )

        # Save the loss for this batch
        running_train_loss += loss.item()
        
        # Calculate training accuracy
        _, predicted = torch.max(outputs, 1)
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

    # Save the loss and accuracy for this epoch
    avg_train_loss = running_train_loss / num_batches
    train_accuracy = correct_train / total_train

    train_loss.append(avg_train_loss)
    train_acc_history.append(train_accuracy)

    # Print the loss for this epoch
    print("-" * 10)
    print(f"Epoch {epoch+1}/{epochs} : Training loss: {train_loss[-1]:.4f}")
    print(f"Epoch {epoch+1}/{epochs} : Training Accuracy: {train_acc_history[-1]:.4f}")

    # VALIDRITON OF EACH EPOCH #
    # Set the model to evaluation mode
    classifier.eval()
    
    num_batches = len(val_loader)
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for batch_idx, data in enumerate(val_loader, start=0):
        
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = classifier(images)
            loss = criterion(outputs, labels)  # For CrossEntropyLoss

            # save the loss for this batch
            running_val_loss += loss.item()
            
            # Calculate validation accuracy
            _, predicted = torch.max(outputs, 1)
            correct_val += (predicted == labels).sum().item()
            total_val += labels.size(0)

        # Save the loss for this epoch
        avg_val_loss = running_val_loss / num_batches
        val_accuracy = correct_val / total_val

        val_loss.append(avg_val_loss)
        val_acc_history.append(val_accuracy)

        # Print the loss for this epoch
        print(f"Epoch {epoch+1}/{epochs} : Validation loss: {val_loss[-1]:.4f}")
        print(f"Epoch {epoch+1}/{epochs} : Validation accuracy: {val_acc_history[-1]:.4f}")
        
    # saving the best model

    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        best_epoch = epoch +1
        torch.save(classifier.state_dict(), os.path.join(output_folder, "best_model.pth"))
        print(f"Best model has been updated...")
     
    # end of epoch time    
    end_time = time.time() 
    elapsed = end_time - start_time
    time_per_epoch.append(elapsed)


# Save the model
torch.save(classifier.state_dict(), os.path.join(output_folder, "last_model.pth"))
print(f"Training completed. Model saved!")
print(f"Best EPOCH: {best_epoch}")

# Save the log
results = ({"epoch": list(range(1, epochs + 1)),
            "time": time_per_epoch,
            "train_accuracy": train_acc_history,
            "val_accuracy": val_acc_history,
            "train_loss": train_loss, 
            "val_loss": val_loss})

results_df = pd.DataFrame(results)
print(results_df.head())

table_save_name = 'training_log'
table_save_name = f'{table_save_name}.csv'
table_save_path = os.path.join(output_folder, table_save_name)
results_df.to_csv(table_save_path, index=False)

# Plot the loss history
plt.plot(train_loss, label="Training loss")
plt.plot(val_loss, label="Validation loss")
plt.legend()
plt.show()

In [ ]:
### LOAD PREVIOUSLY TRAINED MODEL ###

classifier = zooplanktonet_classifier
model_path = ""

classifier.load_state_dict(torch.load(model_path, 
                                      map_location=torch.device(device),))

print(f"Model loaded from files: {model_path}")


In [ ]:
### TEST ###

save = True

classifier.to(device)
classifier.eval()

# test data
class_names = test_dataset.classes  # 17 jellyfish species
samples = test_dataset.samples  # list of (path, class_idx)
results = []  # will hold [filename, true_label, pred_label, confidence]

with torch.no_grad():
    for batch_idx, data in enumerate(test_loader, start=0): #### THIS SHOULD BE test_loader
        
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        
        outputs = classifier(images)  # raw logits
        probs = torch.softmax(outputs, dim=1)           # probabilities
        confs, preds = probs.max(dim=1)                 # best class + confidence

        # compute start index of this batch in the dataset
        start_idx = batch_idx * test_loader.batch_size
        batch_size = images.size(0)
        batch_indices = list(range(start_idx, start_idx + batch_size))

        for idx, true_label_tensor, pred_tensor, conf_tensor in zip(
            batch_indices, labels, preds, confs
        ):
            path, _ = samples[idx]
            filename = os.path.basename(path)  # just filename
            results.append({
                "image": filename,
                "true_label": class_names[true_label_tensor.item()],
                "pred_label": class_names[pred_tensor.item()],
                "confidence": float(conf_tensor.item())
            })


# Create a DataFrame
results_df = pd.DataFrame(results)

if save: 
    table_save_name = 'model_pred'
    table_save_name = f'{table_save_name}.csv'
    table_save_path = os.path.join(output_folder, table_save_name)
    results_df.to_csv(table_save_path, index=False)
    
else:
    print(results_df.head())

def jelly_conf_matrix(data, class_names, class_to_idx, conf_thresh=0.9, save = False):
    """
    Display two confusion matrices side by side:
        Left: counts
        Right: normalized percentages (rounded to 3 digits)
    Args:
        data: pandas DataFrame with ['true_label','pred_label','confidence']
        class_names: list of class names
        class_to_idx: dict mapping class name to numeric index
        conf_thresh: float, filter predictions below this confidence
    """
    # Filter by confidence
    filtered_data = data[data['confidence'] > conf_thresh]

    # Map labels to numeric indices
    y_true = [class_to_idx[label] for label in filtered_data["true_label"]]
    y_pred = [class_to_idx[label] for label in filtered_data["pred_label"]]

    # Compute confusion matrix
    cm_counts = confusion_matrix(y_true, y_pred)
    # Transpose to have x-axis = true labels, y-axis = predicted labels
    cm_counts = cm_counts.T
    
    if save:
        # Save confusion matrix
        plt.figure(figsize=(12, 12))
        disp = ConfusionMatrixDisplay(cm_counts, display_labels=class_names)
        disp.plot(cmap="Blues", xticks_rotation=90, ax=plt.gca())
        plt.xlabel("True Label")
        plt.ylabel("Predicted Label")
        plt.title(f"Confusion Matrix (Counts), Confidence > {conf_thresh}")
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f"conf_matrix_counts.png"), dpi = 300, bbox_inches = "tight")
        plt.close()

    else:
        # Display confusion matrix
        plt.figure(figsize=(8, 8))
        disp = ConfusionMatrixDisplay(cm_counts, display_labels=class_names)
        disp.plot(cmap="Blues", xticks_rotation=90, ax=plt.gca())
        plt.xlabel("True Label")
        plt.ylabel("Predicted Label")
        plt.title(f"Confusion Matrix (Counts), Confidence > {conf_thresh}")
        plt.tight_layout()
        plt.show()

def jelly_conf_matrix_normalized(data, class_names, class_to_idx, conf_thresh=0.9, save = False):
    """
    Display a confusion matrix normalized by the number of images per true class.

    Args:
        data: pandas DataFrame with columns ['true_label','pred_label','confidence']
        class_names: list of class names
        class_to_idx: dict mapping class names to numeric indices
        conf_thresh: float, filter predictions below this confidence
    """
    # Filter by confidence
    filtered_data = data[data['confidence'] > conf_thresh]

    # Map labels to numeric indices
    y_true = [class_to_idx[label] for label in filtered_data["true_label"]]
    y_pred = [class_to_idx[label] for label in filtered_data["pred_label"]]

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Normalize **row-wise** to account for number of images per true class
    cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    # Replace NaNs (for classes with 0 samples) with 0
    cm_normalized = np.nan_to_num(cm_normalized, nan=0.0)

    # Transpose so x-axis = true labels, y-axis = predicted labels
    cm_normalized = cm_normalized.T
    
    if save:
        # Save confusion matrix
        plt.figure(figsize=(12, 12))
        disp = ConfusionMatrixDisplay(cm_normalized, display_labels=class_names)
        disp.plot(cmap="Blues", xticks_rotation=90, ax=plt.gca(), values_format=".0f")
        plt.xlabel("True Label")
        plt.ylabel("Predicted Label")
        plt.title(f"Confusion Matrix (Normalized by True Class, %), Confidence > {conf_thresh}")
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f"conf_matrix_normalized.png"), dpi = 300, bbox_inches = "tight")
        plt.close()

    else: 
        # Display
        plt.figure(figsize=(8, 8))
        disp = ConfusionMatrixDisplay(cm_normalized, display_labels=class_names)
        disp.plot(cmap="Blues", xticks_rotation=90, ax=plt.gca(), values_format=".0f")
        plt.xlabel("True Label")
        plt.ylabel("Predicted Label")
        plt.title(f"Confusion Matrix (Normalized by True Class, %), Confidence > {conf_thresh}")
        plt.tight_layout()
        plt.show()

jelly_conf_matrix(
    results_df, 
    class_names=test_dataset.classes, 
    class_to_idx=test_dataset.class_to_idx,
    conf_thresh=0, 
    save = save)

jelly_conf_matrix_normalized(
    results_df, 
    class_names=test_dataset.classes, 
    class_to_idx=test_dataset.class_to_idx,
    conf_thresh=0, 
    save = save)